In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Compare two DNA sequences (FASTA): length, GC%, Hamming.
Works from CLI, Snakemake, Docker, and Jupyter (no __file__).

Usage:
  python scripts/compare_genes.py \
      -i1 files/human_cytc.fasta \
      -i2 files/mouse_cytc.fasta \
      -o results/compare_genes.tsv
"""
from __future__ import annotations
import argparse
import sys
import os
from pathlib import Path

# -------- Path helpers (robust to Jupyter / frozen / CLI) --------
def get_project_root() -> Path:
    # If __file__ exists → scripts/... → parent of parent = project root
    f = globals().get("__file__", None)
    if f:
        return Path(f).resolve().parents[1]
    # Jupyter: no __file__ — fallback to current working directory
    # (assume notebook started from project root; if нет — укажи абсолютные пути)
    return Path.cwd().resolve()

PROJECT_ROOT = get_project_root()

def as_project_path(p: str | Path) -> Path:
    """Absolute paths stay as is; relative paths are resolved from project root."""
    p = Path(p)
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()

# ---------------- Core logic ----------------
def read_fasta(path: Path) -> str:
    seq = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(">"):
                continue
            # Убираем пробелы внутри строки на всякий случай
            seq.append(line.upper().replace(" ", ""))
    return "".join(seq)

def gc_percent(seq: str) -> float:
    if not seq:
        return 0.0
    gc = seq.count("G") + seq.count("C")
    return 100.0 * gc / len(seq)

def hamming(a: str, b: str) -> int:
    n = min(len(a), len(b))
    return sum(1 for i in range(n) if a[i] != b[i])

def parse_args(argv=None):
    ap = argparse.ArgumentParser(
        description="Compare two DNA sequences: length, GC%%, Hamming distance"
    )
    ap.add_argument("-i1", "--input1", required=True, help="FASTA 1 (DNA)")
    ap.add_argument("-i2", "--input2", required=True, help="FASTA 2 (DNA)")
    ap.add_argument("-o", "--out-tsv", required=True, help="Output TSV path")
    return ap.parse_args(argv)

def main(argv=None) -> int:
    args = parse_args(argv)

    in1 = as_project_path(args.input1)
    in2 = as_project_path(args.input2)
    out = as_project_path(args.out_tsv)

    # Helpful diagnostics
    print(f"[INFO] PROJECT_ROOT = {PROJECT_ROOT}")
    print(f"[INFO] CWD          = {Path.cwd().resolve()}")
    print(f"[INFO] input1       = {in1}")
    print(f"[INFO] input2       = {in2}")
    print(f"[INFO] out-tsv      = {out}")

    # Validate inputs
    missing = [p for p in (in1, in2) if not p.exists()]
    if missing:
        for p in missing:
            print(f"[ERROR] Input not found: {p}", file=sys.stderr)
        return 2

    out.parent.mkdir(parents=True, exist_ok=True)

    s1 = read_fasta(in1)
    s2 = read_fasta(in2)

    L1, L2 = len(s1), len(s2)
    GC1, GC2 = gc_percent(s1), gc_percent(s2)
    hamm = hamming(s1, s2)

    with out.open("w", encoding="utf-8") as w:
        w.write("Gene1\tGene2\tLength1\tLength2\tGC1\tGC2\tHamming\n")
        # В отчёте сохраняем оригинальные аргументы (как ты их передал)
        w.write(f"{args.input1}\t{args.input2}\t{L1}\t{L2}\t{GC1:.2f}\t{GC2:.2f}\t{hamm}\n")

    print(f"[OK] compare_genes → {out}")
    print(f"     Lengths: {L1} vs {L2}")
    print(f"     GC%%:    {GC1:.2f} vs {GC2:.2f}")
    print(f"     Hamming: {hamm} (over min length {min(L1, L2)})")
    return 0

if __name__ == "__main__":
    sys.exit(main())
